In [1]:
# ===== Importing all the required libraries =====
import pandas as pd                      # tables (confusion matrix table)
import numpy as np                       # arrays, argmax
import matplotlib.pyplot as plt          # graphs and images
import seaborn as sns                    # heatmap

from sklearn.metrics import confusion_matrix, classification_report   # checking results

import keras
from keras import layers
from keras.datasets import mnist         # MNIST digit dataset
from keras.utils import to_categorical   # one-hot encoding for labels

In [2]:
# Loading the data: x = images, y = labels (answers)
(x_train, y_train), (x_test, y_test) = mnist.load_data()

In [3]:
# Checking the shape of x_train: (60000, 28, 28) -> no channel, so grayscale
x_train.shape

(60000, 28, 28)

In [4]:
# Reshaping: adding 1 at the end for the grayscale channel (CNN needs 4 values)
x_train = x_train.reshape(60000, 28, 28, 1)
x_test  = x_test.reshape(10000, 28, 28, 1)

In [5]:
# Normalising: dividing pixels by 255 so values go from 0-255 to 0-1
# Only x (images), not y (labels). "/" converts to float automatically
x_train = x_train / 255
x_test  = x_test / 255

In [6]:
# Checking the output variable shape: (60000,) -> plain numbers like 5, 0, 4
y_train.shape

(60000,)

In [7]:
# Converting labels to one-hot encoding: 5 -> [0,0,0,0,0,1,0,0,0,0]
# 10 classes (digits 0-9). 
y_train = to_categorical(y_train, num_classes=10)
y_test  = to_categorical(y_test, num_classes=10)

In [8]:
# ===== Creating the convolution network =====
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Dense, Conv2D, Dropout, Flatten, MaxPooling2D

# Creating an empty model to stack layers one by one
model = Sequential()

# Convolution layer: 28 filters, each 3x3, slides over the image to find patterns
# input_shape = (28, 28, 1) -> 28x28 grayscale image
model.add(Conv2D(28, kernel_size=(3,3), input_shape=(28, 28, 1)))

# Max pooling: takes the highest value from every 2x2 block -> makes the image smaller
model.add(MaxPooling2D(pool_size=(2,2)))

# Flatten: converts the 2D grid into one long row for the Dense layer
model.add(Flatten())

# Dense layer: 128 neurons with ReLU (rectified linear unit) activation
model.add(Dense(128, activation="relu"))

# Dropout: randomly switches off 20% of neurons during training to reduce overfitting
model.add(Dropout(0.2))

# Output layer: 10 neurons (one per digit), softmax gives probabilities that add up to 1
model.add(Dense(10, activation="softmax"))

C:\Users\gayut\anaconda3\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


In [9]:
# Compiling: adam optimizer, categorical_crossentropy loss (because labels are one-hot)
model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=["accuracy"])

# Training for 10 epochs; results saved in "history" to plot later
history = model.fit(x=x_train, y=y_train, epochs=10)

Epoch 1/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 17s 8ms/step - accuracy: 0.9399 - loss: 0.2033
Epoch 2/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9750 - loss: 0.0816
Epoch 3/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9823 - loss: 0.0573
Epoch 4/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9857 - loss: 0.0440
Epoch 5/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9892 - loss: 0.0327
Epoch 6/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9906 - loss: 0.0271
Epoch 7/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9920 - loss: 0.0229
Epoch 8/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9926 - loss: 0.0213
Epoch 9/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9934 - loss: 0.0195
Epoch 10/10
1875/1875 ━━━━━━━━━━━━━━━━━━━━ 15s 8ms/step - accuracy: 0.9936 - loss: 0.0183


In [10]:
# Predicting on the test set: gives 10 probabilities for each image
y_predict = model.predict(x_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step


In [11]:
# Taking the position of the highest probability = predicted digit
y_pred = np.argmax(y_predict, axis=1)

In [12]:
# Showing the predicted digits
y_pred

array([7, 2, 1, ..., 4, 5, 6], shape=(10000,))

In [13]:
# Converting one-hot test labels back to numbers so we can compare with y_pred
y_test_eval = np.argmax(y_test, axis=1)

In [14]:
# Confusion matrix: rows = actual digit, columns = predicted digit
# Diagonal = correct predictions, other cells = mistakes
cm = confusion_matrix(y_test_eval, y_pred)

In [15]:
# Showing the confusion matrix
cm

array([[ 970,    1,    2,    0,    0,    0,    5,    1,    1,    0],
       [   1, 1126,    2,    0,    0,    1,    2,    1,    2,    0],
       [   2,    4, 1013,    0,    1,    0,    1,    9,    2,    0],
       [   0,    0,    1,  997,    0,    3,    0,    3,    1,    5],
       [   0,    1,    0,    0,  962,    0,    5,    0,    0,   14],
       [   3,    0,    0,    4,    0,  876,    3,    0,    3,    3],
       [   4,    3,    0,    1,    1,    2,  944,    0,    3,    0],
       [   1,    3,    5,    0,    1,    0,    0, 1010,    0,    8],
       [   3,    0,    3,    1,    1,    1,    1,    4,  953,    7],
       [   1,    3,    1,    1,    7,    2,    0,    2,    1,  991]])

In [16]:
# Converting the confusion matrix into a table with digit names (0-9)
confusion_matrix_table = pd.DataFrame(cm, index=range(10), columns=range(10))

In [17]:
# Showing the table
confusion_matrix_table

,0,1,2,3,4,5,6,7,8,9
0,970,1,2,0,0,0,5,1,1,0
1,1,1126,2,0,0,1,2,1,2,0
2,2,4,1013,0,1,0,1,9,2,0
3,0,0,1,997,0,3,0,3,1,5
4,0,1,0,0,962,0,5,0,0,14
5,3,0,0,4,0,876,3,0,3,3
6,4,3,0,1,1,2,944,0,3,0
7,1,3,5,0,1,0,0,1010,0,8
8,3,0,3,1,1,1,1,4,953,7
9,1,3,1,1,7,2,0,2,1,991


In [18]:
# Classification report: precision, recall, f1-score for each digit
# Saved as "report" (not "classification_report", or the function name gets overwritten)
from sklearn.metrics import classification_report
report = classification_report(y_test_eval, y_pred)

In [19]:
# Printing the report
print(report)

              precision    recall  f1-score   support

           0       0.98      0.99      0.99       980
           1       0.99      0.99      0.99      1135
           2       0.99      0.98      0.98      1032
           3       0.99      0.99      0.99      1010
           4       0.99      0.98      0.98       982
           5       0.99      0.98      0.99       892
           6       0.98      0.99      0.98       958
           7       0.98      0.98      0.98      1028
           8       0.99      0.98      0.98       974
           9       0.96      0.98      0.97      1009

    accuracy                           0.98     10000
   macro avg       0.98      0.98      0.98     10000
weighted avg       0.98      0.98      0.98     10000



In [20]:
# Testing the model on unseen test data: gives [loss, accuracy]
model.evaluate(x_test, y_test)

313/313 ━━━━━━━━━━━━━━━━━━━━ 2s 4ms/step - accuracy: 0.9842 - loss: 0.0624


[0.062365010380744934, 0.9842000007629395]